# Program 18
## Sentiment Analysis of text reviews using LSTM
1. Read text reviews and clean
2. Tokenize
3. Create sequences (len 10)
4. Apply LSTM with embedding (size 8)
5. Evaluate
6. Test new review

In [2]:
import numpy as np
import pandas as pd
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding

# 1. Provide some sample text reviews and sentiments (1: positive, 0: negative)
data = {
    'review': [
        'I completely loved this movie, it was fantastic!',
        'Terrible performance and awful formatting.',
        'An absolute joy to watch from start to finish.',
        'I hated every second of it. Waste of time.',
        'Wonderful acting, beautiful cinematography.',
        'One of the worst films I have ever seen.',
        'Excellent plot, very engaging and thrilling.',
        'Boring and dull, I fell asleep halfway through.',
        'A true masterpiece of modern cinema.',
        'Horrible direction, the script made no sense.'
    ],
    'sentiment': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuations
    return text

df['cleaned_review'] = df['review'].apply(clean_text)

# 2. Tokenize the dataset
max_words = 1000
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['cleaned_review'])
sequences = tokenizer.texts_to_sequences(df['cleaned_review'])

# 3. Create sequences of length 10
max_length = 10
X = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
y = np.array(df['sentiment'])

print("Sample X (padded sequence):", X[0])
print("Sample y:", y[0])

# 4. Apply LSTM with embedding layer (vector size = 8)
embedding_dim = 8

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_length),
    LSTM(16),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train briefly (using whole dataset for simplicity due to small size)
model.fit(X, y, epochs=15, verbose=0)

# 5. Evaluate the performance
loss, accuracy = model.evaluate(X, y, verbose=0)
print(f"\nModel Evaluation - Accuracy: {accuracy * 100:.2f}%")

# 6. Predict on a new review
new_review = "The story was incredibly engaging and I truly enjoyed watching it!"
cleaned_new = clean_text(new_review)
seq_new = tokenizer.texts_to_sequences([cleaned_new])
X_new = pad_sequences(seq_new, maxlen=max_length, padding='post', truncating='post')

prediction = model.predict(X_new, verbose=0)
print(f"\nNew Review: '{new_review}'")
print(f"Sentiment Prediction: {'Positive' if prediction[0][0] > 0.5 else 'Negative'} ({prediction[0][0]:.4f})")

Sample X (padded sequence): [ 2  8  9 10 11  5 12 13  0  0]
Sample y: 1


c:\Users\vishn\Desktop\Programs\SEM 8\NLP LAB\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Model Evaluation - Accuracy: 90.00%

New Review: 'The story was incredibly engaging and I truly enjoyed watching it!'
Sentiment Prediction: Positive (0.5019)
